# Caribbean Voices ASR - Zindi Hackathon
**Team:** diTranscribers  
**Model:** whisper-large-v3  
**Approach:** Fine-tuning Whisper Large-v3 with LoRA
---

## Section 1: Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
# Install dependencies
!pip install -q transformers peft librosa num2words accelerate tqdm datasets scikit-learn

In [ ]:
# All imports
import os
import re
import pandas as pd
import torch
import librosa
from tqdm import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
from num2words import num2words

# Training imports (used if running Section 2)
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import Dataset
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split

# Reproducibility: Training used random_state=42 for train/val split.
# Inference is deterministic via temperature=0.0 and num_beams=10 (no randomness).

# Extract audio files
!mkdir -p /content/Audio
!unzip -o -q "/content/drive/MyDrive/ZindiHackathon/Audio.zip" -d "/content/Audio"
print(f"Audio files: {len([f for f in os.listdir('/content/Audio') if f.endswith('.wav')])}")

## Section 2: Training

In [ ]:
# Configuration
CONFIG = {
    # Paths
    "data_folder": "/content/drive/MyDrive/ZindiHackathon",
    "audio_folder": "/content/Audio",
    "output_dir": "/content/drive/MyDrive/ZindiHackathon/whisper-augmented-v4",
    "final_model_dir": "/content/drive/MyDrive/ZindiHackathon/whisper-augmented-v4-final",

    # Model
    "base_model": "openai/whisper-large-v3",
    "sampling_rate": 16000,

    # LoRA
    "lora_r": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],

    # Training
    "batch_size": 8,
    "gradient_accumulation_steps": 2,
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "max_steps": 3500,
    "eval_steps": 500,
    "save_steps": 500,
}

print("✓ Config set")

In [ ]:
# Data Collator
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

print("✓ Data collator defined")

In [ ]:
# Load and split data
train_df = pd.read_csv(os.path.join(CONFIG["data_folder"], "Train.csv"))
train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}")

In [ ]:
# Load processor
processor = WhisperProcessor.from_pretrained(CONFIG["base_model"])
print("✓ Processor loaded")

In [ ]:
# Preprocessing function
def preprocess(row):
    audio_path = os.path.join(CONFIG["audio_folder"], f"{row['ID']}.wav")
    audio, _ = librosa.load(audio_path, sr=CONFIG["sampling_rate"])

    input_features = processor.feature_extractor(
        audio, sampling_rate=CONFIG["sampling_rate"], return_tensors="pt"
    ).input_features[0]

    labels = processor.tokenizer(row["Transcription"]).input_ids

    return {"input_features": input_features, "labels": labels}

# Process datasets
print("Processing train...")
train_dataset = Dataset.from_pandas(train_df[["ID", "Transcription"]])
train_dataset = train_dataset.map(preprocess, remove_columns=train_dataset.column_names)

print("Processing val...")
val_dataset = Dataset.from_pandas(val_df[["ID", "Transcription"]])
val_dataset = val_dataset.map(preprocess, remove_columns=val_dataset.column_names)

print("✓ Datasets ready")

In [ ]:
# Load model with LoRA
model = WhisperForConditionalGeneration.from_pretrained(
    CONFIG["base_model"],
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")
model.config.suppress_tokens = []

print("✓ Model loaded with LoRA")

In [ ]:
# Training
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG["output_dir"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    max_steps=CONFIG["max_steps"],
    fp16=True,
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    logging_steps=100,
    report_to=["none"],
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print("🚀 Starting training...")
trainer.train()

In [ ]:
# Save final model
trainer.save_model(CONFIG["final_model_dir"])
processor.save_pretrained(CONFIG["final_model_dir"])
print(f"✓ Model saved to {CONFIG['final_model_dir']}")

## Section 3: Inference

In [ ]:
# Config
ADAPTER_PATH = "/content/FinalModel"
# ADAPTER_PATH = "/content/drive/MyDrive/ZindiHackathon/whisper-augmented-v4-final"
BASE_MODEL = "openai/whisper-large-v3"
TEST_CSV = "/content/drive/MyDrive/ZindiHackathon/Test.csv"
AUDIO_DIR = "/content/Audio"
OUTPUT_PATH = "/content/drive/MyDrive/ZindiHackathon/submission.csv"
SAMPLING_RATE = 16000

print("✓ Config set")

In [ ]:
# Load model
processor = WhisperProcessor.from_pretrained(BASE_MODEL)
print("✓ Processor loaded")

base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)
print("✓ Base model loaded")

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model = model.merge_and_unload()
model.to("cuda").eval()
print("✓ LoRA adapter loaded and merged")

In [ ]:
def normalize_text(text):
    """
    Normalize transcribed text by removing filler words and cleaning formatting.

    Args:
        text: Raw transcription string
    Returns:
        Cleaned lowercase string with fillers and duplicates removed
    """
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r'\buh+\b|\bum+\b|\ber+\b|\bah+\b|\bhmm+\b', '', text)
    text = re.sub(r'\b(\w+)(\s+\1)+\b', r'\1', text)
    text = re.sub(r'[^\w\s\'\-]', '', text)
    return ' '.join(text.split()).strip()

In [ ]:
# Generate predictions
test_df = pd.read_csv(TEST_CSV)
print(f"Test samples: {len(test_df)}")

results = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    audio_id = row["ID"]
    audio_path = os.path.join(AUDIO_DIR, f"{audio_id}.wav")

    try:
        audio, _ = librosa.load(audio_path, sr=SAMPLING_RATE)
        inputs = processor(audio, sampling_rate=SAMPLING_RATE, return_tensors="pt")
        input_features = inputs.input_features.half().to("cuda")

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                num_beams=10,
                temperature=0.0,
                repetition_penalty=1.2,
                max_new_tokens=444,  # Changed from 448 to 444
            )

        text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        text = normalize_text(text)
    except Exception as e:
        print(f"Error on {audio_id}: {e}")
        text = ""

    results.append({"ID": audio_id, "Transcription": text})

# Save
pd.DataFrame(results).to_csv(OUTPUT_PATH, index=False)
print(f"\n✓ Saved to {OUTPUT_PATH}")

In [ ]:
# Preview results
pd.read_csv(OUTPUT_PATH).head(10)

## Section 4: Postprocessing

In [ ]:
def normalize_special_terms(text):
    """
    Convert special terms to spoken form.

    Examples:
        Y2K → year 2000
        G15 → G-Fifteen
        A300 → A-three-hundred
    """
    # Y2K → year 2000
    text = re.sub(r'\bY2K\b', 'year 2000', text, flags=re.IGNORECASE)
    # G15 → G-fifteen (handles any G<number>)
    text = re.sub(
        r'\b[gG](\d+)\b',
        lambda m: f"G-{num2words(int(m.group(1))).replace('-', ' ').capitalize()}",
        text
    )
    # A300 → A-three-hundred (handles uppercase/lowercase A)
    text = re.sub(
        r'\b[aA](\d+)\b',
        lambda m: f"A-{num2words(int(m.group(1))).replace(' ', '-')}",
        text
    )
    return text

def num2words_to_words(text):
    """
    Convert all numbers in text to written words.

    Handles:
        - Decades: 1970s → nineteen seventies
        - Ordinals: 1st → first
        - Standalone numbers: 42 → forty two
    """
    # Decades like 1970s or 70's/70s
    def replace_decade(match):
        num_str = match.group(1)
        num = int(num_str)
        if len(num_str) == 4:
            word = num2words(num // 100).replace('-', ' ') + ' ' + num2words(num % 100).replace('-', ' ')
        else:
            word = num2words(num).replace('-', ' ')
        if word.endswith('y'):
            word = word[:-1] + 'ies'
        else:
            word += 's'
        return word

    text = re.sub(r"\b(\d{2,4})'?s\b", replace_decade, text)

    # Ordinals
    def replace_ordinal(match):
        num_str = match.group(1)
        return num2words(int(num_str), ordinal=True).replace('-', ' ')
    text = re.sub(r'\b(\d+)(st|nd|rd|th)\b', replace_ordinal, text)

    # Standalone numbers
    def replace_number(match):
        num_str = match.group(0)
        return num2words(int(num_str)).replace('-', ' ')
    text = re.sub(r'\b\d+\b', replace_number, text)

    return text

def normalize_final_text(text):
    """
    Final text cleanup: lowercase and collapse whitespace.
    """
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply pipeline to entire DataFrame
df = pd.read_csv(OUTPUT_PATH)
df['Transcription'] = df['Transcription'].apply(
    lambda x: num2words_to_words(normalize_special_terms(x))
)
df["Transcription"] = df["Transcription"].apply(normalize_final_text)

df.to_csv(OUTPUT_PATH, index=False)
print(f"✓ Postprocessing complete. Saved to {OUTPUT_PATH}")